# Analyses of stored results: tests over the sets chosen in advance, tuning budgets, recall of every class, label-free selection with the corrected test

**Computes, without new clustering:** (A) across-set tests (Wilcoxon, sign test, Hodges–Lehmann estimate with exact 95 % interval) of IT2 against FCM and T1d over the ten real sets chosen in advance (the six of notebook 24 and the four of notebook 35) and over all thirteen real sets with at least five classes and a size ratio of at least 10, from `results_30.json` and `results_35.json`. (B) Tuning budget: for every method, the expected held-out ARI when its setting is selected (by ARI on one half of the instances, scored on the other) from $b$ settings drawn at random from its grid, for $b$ from 1 to the grid size; and IT2 at the budget of FCM, of T1d and of the mixture compared with each at its full grid (claim rule; corrected test at $\rho = 1/3$ (n_test/n_train) and at the $\rho$ of the overlap, $f/(1-f)$, which the paper uses). Records of 29 (imbalanced sets), 24 (six sets) and 35 (four sets). (C) Recall of every class, largest first, at the ARI-selected settings on the imbalanced sets (partitions of 29). (D) Label-free selection by the silhouette on the imbalanced sets (records of 29), with the claim rule on the two halves and the corrected test.

**Inputs:** `runs29a_*.pkl` of 29, `runs24_*.pkl` of 24, `runs35_stage2.pkl` of 35, `results_30.json`, `results_35.json`.
**Outputs:** `results/38/results_38.json` and a text report.


In [ ]:
import os, time, json, pickle, platform, glob
import numpy as np, scipy
from scipy import stats
from scipy.optimize import linear_sum_assignment
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.abspath(os.path.join(RESULTS_DIR, "38")); os.makedirs(os.path.join(OUTDIR, "reports"), exist_ok=True)
BASE = os.path.dirname(OUTDIR)
DIR_29 = os.environ.get("DIR_29", os.path.join(BASE, "29")); DIR_24 = os.environ.get("DIR_24", os.path.join(BASE, "24"))
DIR_35 = os.environ.get("DIR_35", os.path.join(BASE, "35"))
J30 = os.environ.get("J30", os.path.join(BASE, "30", "results_30.json")); J35 = os.environ.get("J35", os.path.join(DIR_35, "results_35.json"))
print(f"python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__}")
for nm, d in (("29", DIR_29), ("24", DIR_24), ("35", DIR_35)): print(f"{nm}: {d}  ({len(glob.glob(os.path.join(d, '*.pkl')))} checkpoints)")
print("results_30.json:", os.path.exists(J30), "| results_35.json:", os.path.exists(J35))
MM = "IT2FCM-mm"
NAME = {"FCM": "FCM", "T1dec": "T1d", MM: "IT2", "FCMA": "FCMA", "FCMA-d": "FCMA-d", "csiFCM": "csiFCM", "csiFCM-d": "csiFCM-d",
        "MCFCM": "MC-FCM", "RKM": "RKM-IT2FL", "GMM": "GMM", "T1f": "T1f"}
REGIME = ["imb_C3_r20", "imb_C5_r20", "imb_C8_r20", "ecoli", "dermatology", "hcv"]
SIX = ["zoo", "hcv", "wine_red", "steel", "ctg", "shuttle"]; FOUR = ["wine_white", "covertype", "avila", "anuran"]
NINE = ["ecoli", "yeast", "page_blocks"] + SIX
FRAC = {"page_blocks": 0.30, "shuttle": 0.05, "wine_white": 0.30, "covertype": 0.0025, "avila": 0.07, "anuran": 0.20}
SYNTH = {"imb_C3_r20", "imb_C5_r20", "imb_C8_r20"}
def rho_overlap(nm): f = FRAC.get(nm, 0.75); return 0.0 if nm in SYNTH else f / (1 - f)
def rho_paper(nm):
    if nm in SYNTH: return 0.0
    if nm in FOUR: return rho_overlap(nm)
    return 0.3 / 0.7 if nm == "page_blocks" else 1.0 / 3.0

In [ ]:
# Tests and helpers (the claim rule and the corrected test as in the paper).
def tp_nb(d, rho):
    d = np.asarray(d, float); n = len(d)
    if n < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if n else 0.0, t_p=1.0, w_p=1.0, nb_p=1.0, n=n)
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    se = np.sqrt(d.var(ddof=1) * (1.0 / n + rho)); tnb = d.mean() / se if se > 0 else 0.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, nb_p=float(2 * stats.t.sf(abs(tnb), n - 1)), n=n)
def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."
def verdict_nb(a, b):
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."
def judge(dB, dA, nm):
    """dB: differences on half B (settings from A), dA: on half A; claim rule, corrected test at both values of rho."""
    out = dict(mean=float(np.mean(np.r_[dB, dA])), verdict=verdict(tp_nb(dB, 0), tp_nb(dA, 0)))
    for tag, rho in (("paper", rho_paper(nm)), ("overlap", rho_overlap(nm))):
        a, b = tp_nb(dB, rho), tp_nb(dA, rho); out[f"verdict_nb_{tag}"] = verdict_nb(a, b); out[f"nb_p_{tag}"] = [a["nb_p"], b["nb_p"]]
    return out
def hodges_lehmann(d, conf=0.95):
    d = np.asarray(d, float); n = len(d); W = np.sort([(d[i] + d[j]) / 2 for i in range(n) for j in range(i, n)])
    if n < 6: return float(np.median(W)), float("nan"), float("nan")
    counts = np.zeros(n * (n + 1) // 2 + 1); counts[0] = 1
    for r in range(1, n + 1): counts[r:] = counts[r:] + counts[:-r].copy()
    cdf = np.cumsum(counts) / counts.sum(); k = max(int(np.searchsorted(cdf, (1 - conf) / 2, side="right")), 1)
    return float(np.median(W)), float(W[k - 1]), float(W[len(W) - k])
def across(x):
    x = np.asarray(x, float); hl, lo, hi = hodges_lehmann(x); nz = int((x != 0).sum())
    return dict(n=len(x), positive=int((x > 0).sum()), median=float(np.median(x)), hl=hl, hl_ci=[lo, hi],
                wilcoxon_p=float(stats.wilcoxon(x).pvalue) if nz else 1.0, sign_p=float(stats.binomtest(int((x > 0).sum()), nz).pvalue) if nz else 1.0)
def recall_per_class(y, lab, C):
    Cp = max(C, int(lab.max()) + 1); M = np.zeros((C, Cp)); np.add.at(M, (y, lab), 1)
    r, c = linear_sum_assignment(-M); rec = np.zeros(C)
    for i, j in zip(r, c): rec[i] = M[i, j] / max(M[i].sum(), 1)
    return rec
mark = lambda c: ("*" if c["verdict"] == "BETTER" else ("-" if c["verdict"] == "WORSE" else " ")) + {"BETTER": "b", "WORSE": "w"}.get(c.get("verdict_nb_overlap"), "")
print("helpers ready")

In [ ]:
# Per-instance, per-setting records of the three sources, as {set: {family: {key: array over instances}}} for a
# measure.
def load29(nm, fam):
    p = os.path.join(DIR_29, f"runs29a_{fam}_{nm}.pkl"); return pickle.load(open(p, "rb")) if os.path.exists(p) else None
REC = {}; PART29 = {}
for nm in REGIME:
    for fam in ("FCM", "T1dec", MM, "FCMA", "FCMA-d", "csiFCM", "csiFCM-d", "MCFCM", "RKM", "GMM"):
        R = load29(nm, fam)
        if R is None: continue
        ids = sorted(R); PART29[(nm, fam)] = (ids, R)
        REC.setdefault(nm, {})[fam] = {k: np.array([np.mean(R[s]["rec"][fam][k]["ari"]) for s in ids]) for k in R[ids[0]]["rec"][fam]}
for nm in SIX:
    p = os.path.join(DIR_24, f"runs24_{nm}.pkl")
    if not os.path.exists(p) or nm in REC: continue
    R = pickle.load(open(p, "rb")); ids = sorted(R)
    REC[nm] = {f: {k: np.array([np.mean(R[s]["rec"][f][k]["ari"]) for s in ids]) for k in R[ids[0]]["rec"][f]} for f in ("FCM", "T1dec", MM)}
p35 = os.path.join(DIR_35, "runs35_stage2.pkl")
if os.path.exists(p35):
    O = pickle.load(open(p35, "rb"))
    for nm in FOUR:
        ids = sorted(s for (m, s) in O if m == nm)
        if not ids: continue
        fams = list(O[(nm, ids[0])]["rec"]); REC[nm] = {f: {k: np.array([O[(nm, s)]["rec"][f][k]["ari"] for s in ids]) for k in O[(nm, ids[0])]["rec"][f]} for f in fams}
print({nm: {NAME.get(f, f): len(v) for f, v in d.items()} for nm, d in REC.items()})

In [ ]:
# (A) Across-set tests over the sets chosen in advance and over all real sets with at least five classes and a size
# ratio of at least 10.
d30 = json.load(open(J30)) if os.path.exists(J30) else None; d35 = json.load(open(J35)) if os.path.exists(J35) else None
def per_set_diff(nm, z):
    if d30 and nm in d30["per_set"] and f"IT2 vs {z}" in d30["per_set"][nm]["compare"]: return d30["per_set"][nm]["compare"][f"IT2 vs {z}"]["mean"]
    if d35 and nm in d35["results"] and z in d35["results"][nm]["compare"]: return d35["results"][nm]["compare"][z]["ari"]["mean"]
    return None
GROUPS = {"ten sets chosen in advance": SIX + FOUR, "thirteen regime real sets": NINE + FOUR, "six sets chosen in advance": SIX, "four sets chosen by the filter": FOUR}
RES_A = {}
for g, sets in GROUPS.items():
    RES_A[g] = {}
    for z in ("FCM", "T1dec"):
        x = [per_set_diff(nm, z) for nm in sets]
        if any(v is None for v in x): print(f"{g}: missing per-set results, skipped"); continue
        RES_A[g][z] = dict(sets=sets, diffs=x, **across(x))
        v = RES_A[g][z]; print(f"{g:28s} IT2-{NAME[z]:4s} n={v['n']} positive {v['positive']}  HL {100*v['hl']:+.1f} [{100*v['hl_ci'][0]:+.1f}, {100*v['hl_ci'][1]:+.1f}]  Wilcoxon p={v['wilcoxon_p']:.3f}  sign p={v['sign_p']:.3f}")

In [ ]:
# (B) Tuning budget: expected held-out ARI when the setting is selected from b settings drawn at random from the grid.
N_DRAW = int(os.environ.get("N_DRAW38", "500"))
def heldout(vals, keys):
    """Per-instance held-out values (half B scored at the key selected on A, half A at the key selected on B)."""
    n = len(next(iter(vals.values()))); h = n // 2; A, B = np.arange(h), np.arange(h, n)
    kA = max(keys, key=lambda k: vals[k][A].mean()); kB = max(keys, key=lambda k: vals[k][B].mean())
    out = np.empty(n); out[B] = vals[kA][B]; out[A] = vals[kB][A]; return out
def expected_heldout(vals, b, seed=0):
    keys = list(vals); rng = np.random.default_rng(seed)
    if b >= len(keys): return heldout(vals, keys)
    return np.mean([heldout(vals, [keys[i] for i in rng.choice(len(keys), b, replace=False)]) for _ in range(N_DRAW)], axis=0)
BUDGETS = (1, 3, 7, 9, 16, 27, 49, 63, 81, 108, 243)
RES_B = {}; t0 = time.time()
for nm, fams in REC.items():
    n = len(next(iter(fams[MM].values()))); h = n // 2
    curves = {NAME.get(f, f): {b: float(expected_heldout(v, b).mean()) for b in BUDGETS if b <= len(v)} for f, v in fams.items()}
    matched = {}
    for z in ("FCM", "T1dec", "GMM", "FCMA-d", "csiFCM-d", "T1f"):
        if z not in fams: continue
        b = len(fams[z]); xI = expected_heldout(fams[MM], b, seed=1); xZ = heldout(fams[z], list(fams[z]))
        d = xI - xZ; matched[f"IT2 at {b} vs {NAME.get(z, z)}"] = judge(d[h:], d[:h], nm)
    full = {}
    for z in ("FCM", "T1dec"):
        d = heldout(fams[MM], list(fams[MM])) - heldout(fams[z], list(fams[z])); full[f"IT2 full grid ({len(fams[MM])}) vs {NAME[z]}"] = judge(d[h:], d[:h], nm)
    RES_B[nm] = dict(n=n, grid={NAME.get(f, f): len(v) for f, v in fams.items()}, curves=curves, matched=matched, full=full)
    print(f"{nm:12s} " + "  ".join(f"{k} {100*c['mean']:+.1f}{mark(c)}" for k, c in {**full, **matched}.items()) + f" ({time.time()-t0:.0f}s)", flush=True)

In [ ]:
# (C) Recall of every class (largest first) at the ARI-selected settings, partitions of 29.
RES_C = {}
for nm in REGIME:
    if (nm, MM) not in PART29: continue
    ids, R0 = PART29[(nm, MM)]; n = len(ids); h = n // 2; A, B = ids[:h], ids[h:]
    sizes = np.bincount(R0[ids[0]]["y"].astype(int)); order = np.argsort(-sizes); res = dict(sizes=sizes[order].tolist(), recall={})
    for fam in ("FCM", "T1dec", MM, "FCMA-d", "csiFCM-d", "MCFCM", "GMM"):
        if (nm, fam) not in PART29: continue
        ids_f, R = PART29[(nm, fam)]; vals = REC[nm][fam]; keys = list(vals); idx = {s: i for i, s in enumerate(ids_f)}
        kA = max(keys, key=lambda k: np.mean([vals[k][idx[s]] for s in A])); kB = max(keys, key=lambda k: np.mean([vals[k][idx[s]] for s in B]))
        recs = []
        for s in ids_f:
            k = kA if s in B else kB; y = R[s]["y"].astype(int); C = R[s]["C"]
            recs.append(np.mean([recall_per_class(y, np.asarray(l).astype(int), C) for l in R[s]["rec"][fam][k]["lab"]], axis=0))
        res["recall"][NAME[fam]] = np.mean(recs, axis=0)[order].tolist()
    RES_C[nm] = res
    print(f"{nm:12s} sizes {res['sizes']}\n" + "\n".join(f"   {f:8s} " + " ".join(f"{100*v:4.0f}" for v in r) for f, r in res["recall"].items()))

In [ ]:
# (D) Label-free selection by the silhouette (records of 29): every method takes, on each instance, the setting with
# the largest mean silhouette.
RES_D = {}
for nm in REGIME:
    if (nm, MM) not in PART29: continue
    ids, _ = PART29[(nm, MM)]; n = len(ids); h = n // 2; val = {}
    for fam in ("FCM", "T1dec", MM, "FCMA-d", "csiFCM-d", "MCFCM", "GMM"):
        if (nm, fam) not in PART29: continue
        ids_f, R = PART29[(nm, fam)]
        val[fam] = np.array([float(np.mean(R[s]["rec"][fam][max(R[s]["rec"][fam], key=lambda k: np.mean(R[s]["rec"][fam][k]["sil"]))]["ari"])) for s in ids_f])
    RES_D[nm] = dict(level={NAME[f]: float(v.mean()) for f, v in val.items()},
                     compare={NAME[z]: judge((val[MM] - val[z])[h:], (val[MM] - val[z])[:h], nm) for z in val if z != MM})
    print(f"{nm:12s} silhouette: " + "  ".join(f"IT2-{z} {100*c['mean']:+.1f}{mark(c)}" for z, c in RES_D[nm]["compare"].items()))

In [ ]:
# Report and result file.
LINES = []
def say(s=""): print(s); LINES.append(s)
def cell(c): return f"{100*c['mean']:+6.1f}{mark(c):3s}" + {"BETTER": "b", "WORSE": "w"}.get(c.get("verdict_nb_paper"), "")
say(f"NOTEBOOK 38  run {RUN_STAMP}")
say("(A) ACROSS-SET TESTS, IT2 MINUS FCM AND T1d (label-tuned ARI x100, per-set means)")
for g, d in RES_A.items():
    for z, v in d.items():
        say(f"  {g:28s} IT2-{NAME[z]:4s} n={v['n']:2d} positive {v['positive']:2d}  median {100*v['median']:+.1f}  HL {100*v['hl']:+.1f} [{100*v['hl_ci'][0]:+.1f}, {100*v['hl_ci'][1]:+.1f}]  Wilcoxon p={v['wilcoxon_p']:.4f}  sign p={v['sign_p']:.3f}")
        say("      " + ", ".join(f"{nm} {100*x:+.1f}" for nm, x in zip(v["sets"], v["diffs"])))
say("(B) TUNING BUDGET: IT2 SELECTED FROM AS MANY SETTINGS AS EACH COMPETITOR HAS (expected over random draws), ARI x100;")
say("    * / - claim rule; then b/w corrected test at the overlap rho, then at rho = n_test/n_train")
for nm, r in RES_B.items():
    say(f"  {nm:12s} grids {r['grid']}")
    say("     " + "  ".join(f"{k}: {cell(c)}" for k, c in {**r['full'], **r['matched']}.items()))
    say("     expected held-out ARI by budget: " + "; ".join(f"{f} " + " ".join(f"{b}:{100*v:.1f}" for b, v in cv.items()) for f, cv in r["curves"].items()))
say("(C) RECALL OF EVERY CLASS (%), CLASSES FROM LARGEST TO SMALLEST, AT THE ARI-SELECTED SETTINGS")
for nm, r in RES_C.items():
    say(f"  {nm}  class sizes (instance 0) {r['sizes']}")
    for f, v in r["recall"].items(): say(f"     {f:8s} " + " ".join(f"{100*x:4.0f}" for x in v))
say("(D) LABEL-FREE SELECTION BY THE SILHOUETTE, IT2 MINUS EACH (ARI x100)")
for nm, r in RES_D.items():
    say(f"  {nm:12s} " + "  ".join(f"{z}: {cell(c)}" for z, c in r["compare"].items()) + "   levels " + ", ".join(f"{k} {100*v:.1f}" for k, v in r["level"].items()))
payload = dict(notebook="38", run_stamp=RUN_STAMP, across=RES_A, budget=RES_B, recall=RES_C, silhouette=RES_D, n_draw=N_DRAW)
for p in (os.path.join(OUTDIR, "results_38.json"), os.path.join(OUTDIR, "results_38_report.txt"), os.path.join(OUTDIR, "reports", f"results_38_report_{RUN_STAMP}.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1, default=str) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n"))
print("saved", os.path.join(OUTDIR, "results_38.json"))